# 03 Compressor characterization

Analyzes a session captured with `uv run pedalrig pedal --model compressor`. Every capture is referenced to the cable
loopback (gain, polarity, same-signal reference) from the calibration session in `meta.json` → `calibration_ref`.
Levels in dBu are send-referred (0 dBFS sine peak = +14.5 dBu). Sections skip themselves when their captures are absent.

A compressor has memory, so level tests hold each level until it settles (staircase) and timing tests use abrupt
20 dB jumps between −25.5 and −5.5 dBu (soft playing ↔ hard strum).

In [ ]:
%matplotlib inline
%load_ext autoreload
# pick up edits to pedalrig without restarting the kernel
%autoreload 2
import os
from pathlib import Path

import numpy as np

if Path.cwd().name == "notebooks":  # sessions/ lives in the project root
    os.chdir("..")

from pedalrig import analysis as A
from pedalrig import plots as P
from pedalrig.compressor import plots as CP
from pedalrig.compressor.analysis import CompressorAnalysis
from pedalrig.models import PEDAL_MODELS, latest_pedal_session
from pedalrig.session import Session, latest_calibration

In [ ]:
ps_path = latest_pedal_session("compressor")     # or: ps_path = Path("sessions/<dir>")
ps = Session(ps_path)
cs = Session(ps.meta["calibration_ref"] or latest_calibration())
cal = A.calibration_summary(cs)
ca = CompressorAnalysis(ps, cal, cs)
have = set(ps.capture_names())
model = PEDAL_MODELS[ps.meta["pedal_model"]]
POS = ("min", "noon", "max")
print("pedal session:", ps_path)
print("calibration:  ", cs.path)
print(f"pedal model:   {ps.meta['pedal_model']} (knobs {', '.join(model.knobs)})")
print(f"G = {cal['G_db']:.3f} dB, polarity {cal['polarity']:+d}, harmonics valid to {ca.f_valid_hi:.0f} Hz")
for c in ps.meta["captures"]:
    flags = " ".join(k for k in ("clipped", "xrun", "kept_with_warning") if c[k])
    print(f"  {c['name']:<26} peak {c['peak_dbfs'][0]:6.1f} dBFS  {flags}")


def fmt(v, unit="", nd=1):
    return "n/a" if v is None else (f"{v:.{nd}f}{unit}" if np.isfinite(v) else f"inf{unit}")

## C1 — From which level does it compress, and how strongly? (1 kHz staircase)

Each 3 dB step is held 2 s; the last 0.5 s is measured. **Threshold** = input where the gain has dropped 1 dB below
the low-level gain. **Ratio** = 1 / slope of output vs input at least 6 dB above the threshold (past the knee).
**Limit**: if the curve turns back toward 1:1 at the top, the gain reduction has run out; the ratio is fitted below that.
`settle` > 0.3 dB means the output was still moving at the end of a step.

In [ ]:
static = {p: ca.static_curve(f"T1_steps_1k_comp-{p}") for p in POS if f"T1_steps_1k_comp-{p}" in have}
for p, r in static.items():
    r["label"] = f"{model.comp_knob}={p}"
    print(f"{r['label']:<10} low-level gain {r['low_gain_db']:+.1f} dB (slope {r['low_slope']:.2f})  "
          f"threshold {fmt(r['threshold_dbu'], ' dBu')}  ratio {fmt(r['ratio'], ':1')}  "
          f"max reduction {r['max_gr_db']:.1f} dB"
          + (f" (limit: 1:1 again above {r['runout_dbu']:+.1f} dBu)" if r["runout_dbu"] is not None else "")
          + f"  output span over top 30 dB of input {r['out_span_top30_db']:.1f} dB  "
          f"settle {r['max_settle_db']:.2f} dB")
if static:
    _ = CP.plot_static(list(static.values()))
    print("\nlocal slope (1 = no compression, 1/ratio above the knee), noon:")
    r = static.get("noon")
    if r is not None:
        mids = (r["in_dbu"][1:] + r["in_dbu"][:-1]) / 2
        print("  " + "  ".join(f"{m:+.0f}:{s:.2f}" for m, s in zip(mids, r["local_slope"])))
else:
    print("no T1 staircase captures")

## C2 — How fast does it react and recover? (20 dB jumps, 3 repeats)

Gain = output − input per cycle (1 ms at 1 kHz). **Attack** = time after the jump up until the gain has covered 63 % / 90 %
of its change; **release** the same after the jump down. **Overshoot** = how far the first 50 ms after the jump up exceed
the settled loud level (the "click" of a slow attack). **Gain dip** = how far the gain swings below its settled value
after the attack (over-compression, then a slight swell back). Spread = max − min over the 3 repeats.

In [ ]:
steps_res = {p: ca.level_steps_response(f"T1_attack_1k_comp-{p}") for p in POS if f"T1_attack_1k_comp-{p}" in have}
for p, r in steps_res.items():
    r["label"] = f"{model.comp_knob}={p}"
    s = ca.timing_summary(r)
    ms = lambda v: fmt(None if v is None else v * 1e3, " ms", 0)  # noqa: E731
    print(f"{r['label']:<10} attack: gain change {fmt(s['attack_gain_change_db'], ' dB')}, t63 {ms(s['attack_t63_s'])}, "
          f"t90 {ms(s['attack_t90_s'])} (spread {ms(s['attack_t63_s_spread'])}), overshoot {fmt(s['attack_overshoot_db'], ' dB')}, "
          f"gain dip after the attack {fmt(s['attack_dip_db'], ' dB')} at {ms(s['attack_dip_s'])}")
    print(f"{'':<10} release: gain change {fmt(s['release_gain_change_db'], ' dB')}, t63 {ms(s['release_t63_s'])}, "
          f"t90 {ms(s['release_t90_s'])} (spread {ms(s['release_t63_s_spread'])}); "
          f"still moving at segment end: {max(abs(t['settle_db']) for t in r['transitions']):.2f} dB")
if steps_res:
    _ = CP.plot_steps(list(steps_res.values()))
else:
    print("no T1 jump captures")

## C3 — Does recovery depend on how long the loud part lasted? (Comp noon)

Release after a 0.1 s and after a 2 s loud note. Equal times = fixed release; a longer release after the long note =
program-dependent release (typical of optical designs).

In [ ]:
if "T1_hold_1k_comp-noon" in have:
    hold = ca.level_steps_response("T1_hold_1k_comp-noon")
    for t in hold["transitions"]:
        if t["kind"] == "release":
            print(f"release after {t['hold_s'] * 1e3:5.0f} ms loud: gain change {t['gain_change_db']:+.1f} dB, "
                  f"t63 {fmt(None if t['t63_s'] is None else t['t63_s'] * 1e3, ' ms', 0)}, "
                  f"t90 {fmt(None if t['t90_s'] is None else t['t90_s'] * 1e3, ' ms', 0)}")
    _ = CP.plot_hold(hold)
else:
    print("no hold capture")

## C9 — How does it change a decaying note? (200 Hz, −1.5 dBu attack, 10 dB/s decay)

The input falls 20 dB in 2 s. A compressor makes the output fall more slowly while the note is above the threshold
(sustain) and may let the attack through before it reacts (`attack peak`, re the level at 0.1 s).

In [ ]:
decay = {p: ca.decay_response(f"T1_pluck_200_comp-{p}") for p in POS if f"T1_pluck_200_comp-{p}" in have}
for p, r in decay.items():
    r["label"] = f"{model.comp_knob}={p}"
    print(f"{r['label']:<10} output decay {r['out_rate_db_s']:+.1f} dB/s (input {r['in_rate_db_s']:+.1f}), "
          f"20 dB down after {fmt(r['out_time_20db_s'], ' s', 2)} (input {fmt(r['in_time_20db_s'], ' s', 2)}), "
          f"attack peak {r['attack_peak_db']:+.1f} dB")
if decay:
    _ = CP.plot_decay(list(decay.values()))
else:
    print("no pluck captures")

## C7 — How much noise does it add? (no input, Volume noon)

RMS 20 Hz–20 kHz at the pedal output, in dBu, vs the loopback's own floor. Compressors raise their own noise with the
makeup gain, so it usually grows with the Comp setting.

In [ ]:
noise = {p: ca.noise(f"T1_noise_comp-{p}") for p in POS if f"T1_noise_comp-{p}" in have}
for p, r in noise.items():
    r["label"] = f"{model.comp_knob}={p}"
    print(f"{r['label']:<10} output noise {r['noise_dbu']:.1f} dBu (loopback floor {r['floor_dbu']:.1f} dBu), "
          f"worst hum line {max(r['hum'].values()):.0f} dBFS")
if noise:
    _ = CP.plot_noise(list(noise.values()))
else:
    print("no noise captures")

## C6 — Distortion while compressing (Comp max, 100 Hz vs 1 kHz)

Fast compressors distort bass: the gain moves within each slow cycle. Compare THD of the settled steps at 100 Hz and
1 kHz, and whether the compression curve itself differs with frequency.

In [ ]:
if {"T1_steps_100_comp-max", "T1_steps_1k_comp-max"} <= have:
    bass = ca.static_curve("T1_steps_100_comp-max")
    bass["label"] = f"{model.comp_knob}=max, 100 Hz"
    hi = dict(static["max"], label=f"{model.comp_knob}=max, 1 kHz")
    for r in (bass, hi):
        print(f"{r['label']:<18} threshold {fmt(r['threshold_dbu'], ' dBu')}, ratio {fmt(r['ratio'], ':1')}, "
              f"THD at the top 3 steps " + ", ".join(f"{v:.1f}" for v in r["thd_db"][-3:]) + " dB")
    _ = CP.plot_static([bass, hi], "Comp max: 100 Hz vs 1 kHz")
else:
    print("no 100 Hz staircase")

## C4 — Is compression the same for every frequency? (all knobs noon)

Gain reduction vs frequency = small-signal (−50 dBFS) response − compressed response, for a constant-level sweep.
Flat = the level detector treats all frequencies alike. Larger reduction at some frequencies = the detector is more
sensitive there (e.g. a bass-heavy note pumps more).

In [ ]:
names = {lv: f"T2_sweep_{lv}_tone-noon" for lv in ("m50", "m30", "m10")}
if all(n in have for n in names.values()):
    gq = ca.compression_vs_freq(names["m50"], names["m30"])
    gl = ca.compression_vs_freq(names["m50"], names["m10"])
    F = [50, 100, 200, 500, 1000, 2000, 5000, 10000]
    for lab, g in (("−15.5 dBu", gq), ("+4.5 dBu", gl)):
        print(f"gain reduction at {lab:>9}: " + " ".join(f"{f}:{np.interp(f, g['f'], g['gr_db']):5.1f}" for f in F) + " dB")
    _ = P.plot_fr([(f"{lv}", r["f"], r["mag_db"], None) for lv, r in
                   (("small-signal −50 dBFS", gl["small"]), ("quiet −30 dBFS", gq["loud"]), ("loud −10 dBFS", gl["loud"]))],
                  title="Pedal response at three input levels (re loopback), all knobs noon")
    _ = CP.plot_gr_vs_freq([("at −15.5 dBu in", gq["f"], gq["gr_db"]), ("at +4.5 dBu in", gl["f"], gl["gr_db"])])
else:
    print("no complete T2 Tone=noon captures")

## C5 — What does Tone do, and does it act before the level detector?

A filter after the compressor changes the response by the same dB at any level. A filter before the detector is
partly undone by the compression at the loud level (a boost raises the detected level, so the gain drops).

In [ ]:
lvs = ("m50", "m30", "m10")
if all(f"T2_sweep_{lv}_tone-{p}" in have for lv in lvs for p in POS):
    res = {p: {lv: ca.sweep_response(f"T2_sweep_{lv}_tone-{p}") for lv in lvs} for p in POS}
    _ = P.plot_fr([(f"Tone={p}", r["m50"]["f"], r["m50"]["mag_db"], r["m50"]["phase_deg"]) for p, r in res.items()],
                  title="Tone at small signal (re loopback)")
    F = np.array([100, 200, 500, 1000, 2000, 3000, 5000, 10000])
    for p in ("min", "max"):
        f = res[p]["m50"]["f"]
        dq = np.interp(F, f, res[p]["m50"]["mag_db"] - res["noon"]["m50"]["mag_db"])
        dl = np.interp(F, f, res[p]["m10"]["mag_db"] - res["noon"]["m10"]["mag_db"])
        print(f"Tone={p} re noon at {F.tolist()} Hz:\n   small-signal {np.round(dq, 1)}\n   loud         {np.round(dl, 1)}"
              f"\n   max |difference| {np.max(np.abs(dl - dq)):.1f} dB")
    print("same change at both levels (<= 1 dB) = after the compressor; smaller change when loud = before the detector")
else:
    print("no complete T2 captures")

## C8 — Volume check

In [ ]:
vn = [f"T3_sweep_m30_vol-{p}" for p in ("9", "noon", "3")]
if all(n in have for n in vn):
    vc = ca.volume_check(vn)
    for n, m in vc["metrics"].items():
        print(f"{n:<24} level re noon {m['level_re_noon_db']:+6.2f} dB   shape deviation {m['shape_dev_db']:.2f} dB")
    _ = P.plot_fr([(n.rsplit("-", 1)[1], r["f"], r["mag_db"], None) for n, r in vc["responses"].items()],
                  title="Volume (re loopback)")
else:
    print("no T3 captures")

## Reading the results for chain placement

- **Threshold in dBu vs your playing level** (single coils roughly −30…−10 dBu): a threshold below your softest playing
  compresses everything (sustain, even dynamics); above it, only peaks are caught.
- **Ratio**: ~2:1 is gentle levelling, 4:1 clearly squashed, ∞:1 a limiter.
- **Attack**: a few ms clamps the pick attack; 20–50 ms lets it through (punchier). **Release**: short = recovers between
  notes but can pump; long = smooth but holds the level down after a loud chord.
- **Noise** grows with the Comp setting: what comes before it (fuzz, high gain) gets its noise floor raised too.
- **Before or after drive**: before an overdrive, an even level makes the drive's clipping more uniform; after it, the
  compressor mostly levels the drive's output, which is already compressed by the clipping.